# mc-pi: 推定誤差のサンプル数依存性

`data/runs/mc-pi/` のラン（`experiments/mc-pi/mc-pi.py` の出力）を集計し，推定誤差が $1/\sqrt{N}$ で減ることを確認する。

- 入力: `data/runs/mc-pi/<hash>/{input,output}.npz`
- キャッシュ: `cache/mc-pi.npz`（`refresh` で更新タイミングを明示する。`src/run_cache.py` 参照）
- 出力: `figs/`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from src.run_cache import load_runs_cached
from src.runs import REPO_ROOT, RUNS_ROOT, load_run

DATA_DIR = RUNS_ROOT / "mc-pi"
CACHE_PATH = REPO_ROOT / "cache" / "mc-pi.npz"
FIG_DIR = Path("figs")
FIG_DIR.mkdir(exist_ok=True)

In [ ]:
# loader はモジュールのトップレベル関数（src.runs.load_run）を使う。
# ノートブック内で定義した関数は，spawn 方式の並列実行（macOS のデフォルト）では pickle できない。
df = load_runs_cached(DATA_DIR, load_run, CACHE_PATH, refresh="new")
df["abs_error"] = (df["pi_hat"] - np.pi).abs()
df.head()

In [ ]:
summary = df.groupby("n_samples").agg(
    n_runs=("seed", "size"),
    mean_abs_error=("abs_error", "mean"),
    mean_std_error=("std_error", "mean"),
)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
ax.loglog(summary.index, summary["mean_abs_error"], "o", label="mean |error|")
ax.loglog(summary.index, summary["mean_std_error"], "--", label="standard error")
ax.set_xlabel("$N$ (samples)")
ax.set_ylabel(r"$|\hat{\pi} - \pi|$")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "error_vs_n.pdf")